In [ ]:
import xarray as xr
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import cmocean as cmo
import cmcrameri as cmc
from scipy import stats
from scipy.optimize import curve_fit
from scipy.stats import binned_statistic_2d

import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
source = 'hofsteenge'

In [ ]:
ds = xr.open_dataset('/home/erwin/data/dalum_racmo241/ANT11_masks.nc')
mask = ds.IceMask
topo = ds.Topography.values.squeeze()
lat = ds.lat.values.squeeze()
lon = ds.lon.values.squeeze()
ds.close()

ds = xr.open_mfdataset(f'/home/erwin/data/{source}/subltot_*.nc')
subl = -ds.subltot.isel(height=0)
ds.close()

In [ ]:
def haversine_distance(lon1, lat1, lon2, lat2):
    """
    Calculate great-circle distance between two points using Haversine formula.
    Returns distance in meters.
    """
    R = 6371000  # Earth's radius in meters
    
    # Convert to radians
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

def compute_slope(topography, lat, lon):
    """
    Compute slope (unitless, meters/meters) on a rotated grid.
    
    Parameters
    ----------
    topography : ndarray, shape (rlat, rlon)
        Elevation data
    lat : ndarray, shape (rlat, rlon)
        Latitude coordinates
    lon : ndarray, shape (rlat, rlon)
        Longitude coordinates
    
    Returns
    -------
    slope : ndarray, shape (rlat, rlon)
        Slope magnitude in meters/meters
    """
    
    # Calculate gradients in grid space (differences between adjacent grid points)
    # Using central differences: gradient at each point based on neighbors
    dz_drlon = np.gradient(topography, axis=1)  # dz/d(rlon)
    dz_drlat = np.gradient(topography, axis=0)  # dz/d(rlat)
    
    # Calculate distances in the rlon direction (constant rlat)
    # For each row, calculate distance to the next column
    dx = np.zeros_like(lon, dtype=float)
    dx[:, :-1] = haversine_distance(lon[:, :-1], lat[:, :-1], 
                                      lon[:, 1:], lat[:, 1:])
    # Extend the last column (use same distance as previous)
    dx[:, -1] = dx[:, -2]
    
    # Calculate distances in the rlat direction (constant rlon)
    # For each column, calculate distance to the next row
    dy = np.zeros_like(lat, dtype=float)
    dy[:-1, :] = haversine_distance(lon[:-1, :], lat[:-1, :],
                                      lon[1:, :], lat[1:, :])
    # Extend the last row (use same distance as previous)
    dy[-1, :] = dy[-2, :]
    
    # Convert grid gradients to physical gradients (meters/meter)
    # dz/dx and dz/dy are in units of [meters]/[meters] = unitless
    dz_dx = dz_drlon / dx  # Change in elevation per meter in x direction
    dz_dy = dz_drlat / dy  # Change in elevation per meter in y direction
    
    # Compute slope magnitude: sqrt((dz/dx)^2 + (dz/dy)^2)
    slope = np.sqrt(dz_dx**2 + dz_dy**2)
    
    return slope


In [ ]:
# Compute slope and convert to xarray of same dimensions as subl

slope = compute_slope(topo,lat,lon)

ref = subl
Hs_slope = xr.DataArray(
    data=np.broadcast_to(slope[None, :, :], ref.shape),  # Add time axis
    dims=['time', 'rlat', 'rlon'],
    coords={
        'time': ref.time.values,
        'rlat': ref.rlat.values,
        'rlon': ref.rlon.values,
    },
    attrs={
        'long_name': 'Topography slope',
        'units': 'dimensionless',
        'standard_name': 'surface_slope'
    }
)

Hs_slope = Hs_slope.chunk({'time':1})

## Sublimation

In [ ]:
ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/tas_*.nc')
tas = ds.tas.isel(height=0)
ds.close()

ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/sfcwind_*.nc')
u10abs = ds.sfcwind.isel(height=0)
ds.close()

ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/subltot_*.nc')
subl = -ds.subltot.isel(height=0)
ds.close()

subl = subl.where(mask, np.nan).values / 1000 # kg/m2/month to mwe/month
u10abs = u10abs.values
tas = tas.values

valid = ~np.isnan(subl) & ~np.isnan(u10abs) &~np.isnan(tas)
subl_valid = subl[valid]
u10abs_valid = u10abs[valid]
tas_valid = tas[valid]
slope_valid = Hs_slope.values[valid]

In [ ]:
fig,ax = plt.subplots(1,3, figsize=(15,5),sharey=True)

h, xedges, yedges, im = ax[0].hist2d(
    u10abs_valid, subl_valid,
    bins=(300, 300),          # Resolution
    cmap='cmc.batlow_r',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6),
    vmin=None               # Auto-scaling
)

h, xedges, yedges, im = ax[1].hist2d(
    tas_valid, subl_valid,
    bins=(300, 300),          # Resolution
    cmap='cmc.batlow_r',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6),
    vmin=None               # Auto-scaling
)

h, xedges, yedges, im = ax[2].hist2d(
    slope_valid, subl_valid,
    bins=(300, 300),          # Resolution
    cmap='cmc.batlow_r',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6),
    vmin=None               # Auto-scaling
)

In [ ]:
def plot_2dhist(ax,xarr,yarr,zarr):

    xbins = np.linspace(xarr.min(), xarr.max(), 50)
    ybins = np.linspace(yarr.min(), yarr.max(), 50)

    statistic, x_edges, y_edges, binnumber = binned_statistic_2d(
        xarr, yarr, zarr,
        statistic='mean',  # or 'mean', 'sum', 'std'
        bins=[xbins, ybins],
        expand_binnumbers=False
    )

    mesh = ax.pcolormesh(
        x_edges, y_edges,
        statistic.T,  # Transpose to match axis orientation
        cmap='RdBu_r',
        vmin=-.1,vmax=.1
    )
    plt.colorbar(mesh,ax=ax)
    ax.contour(x_edges[:-1],y_edges[:-1],statistic.T,20,linewidths=.2,colors='k')
    ax.contour(x_edges[:-1],y_edges[:-1],statistic.T,levels=[0.,1000.],linewidths=1.,colors='k')

    return

fig, ax = plt.subplots(2,2,figsize=(10,10))

plot_2dhist(ax[0,0],tas_valid,u10abs_valid,subl_valid)
plot_2dhist(ax[1,0],tas_valid,u10abs_valid*slope_valid**.5,subl_valid)
#plot_2dhist(ax[1,1],tas_valid,slope_valid*u10abs_valid**2,subl_valid)
plot_2dhist(ax[0,1],u10abs_valid,slope_valid,subl_valid)

subl_fake = 2.3e-3*u10abs_valid**2.3*slope_valid
plot_2dhist(ax[1,1],u10abs_valid,slope_valid,subl_fake)


In [ ]:
def wind_slope(data, a, b, c):
    U,S = data
    return a*U**b*S**c

popt_ws, pcov = curve_fit(wind_slope,
                       (u10abs_valid,slope_valid), 
                       subl_valid, 
                       p0=[0.01, 2, .5])

print(popt_ws)

In [ ]:
def linmod(x,a):
    return a*x

xarr = tas_valid
yarr = u10abs_valid**popt_ws[1] * slope_valid**popt_ws[2]

xbins = np.linspace(xarr.min(), xarr.max(), 100)
ybins = np.linspace(yarr.min(), yarr.max(), 100)

statistic, x_edges, y_edges, binnumber = binned_statistic_2d(
    xarr, yarr, subl_valid,
    statistic='mean',  # or 'mean', 'sum', 'std'
    bins=[xbins, ybins],
    expand_binnumbers=False
)

fig, ax = plt.subplots(1,3,figsize=(10,4))
mesh = ax[0].pcolormesh(
    x_edges, y_edges,
    statistic.T,  # Transpose to match axis orientation
    cmap='RdBu_r',
    vmin=-.2,vmax=.2
)
ax[0].contour(x_edges[:-1],y_edges[:-1],statistic.T,20,linewidths=.2,colors='k')
plt.colorbar(mesh,orientation='horizontal')

cmap = plt.get_cmap('cmo.thermal')
for y,yy in enumerate(ybins[:-1]):
    ax[1].plot(xbins[:-1],statistic[:,y],c=cmap((yy-ybins[0])/(ybins[-1]-ybins[0])))

popts = np.zeros(len(xbins)-1)
popts2 = np.zeros(len(xbins)-1)

for x,xx in enumerate(xbins[:-1]):
    ax[2].plot(ybins[:-1],statistic[x,:],c=cmap((xx-xbins[0])/(xbins[-1]-xbins[0])))
    valid = ~np.isnan(ybins[:-1]) & ~np.isnan(statistic[x,:])

    popt, pcov = curve_fit(linmod, 
    ybins[:-1][valid], statistic[x,:][valid], p0=.001)
    #ybins[:-1][valid], statistic[x,:][valid], p0=[.001,1],bounds=([0,-10],[1,10]))
    popts[x] = popt[0]
    #popts2[x] = popt[1]
    ax[2].plot(ybins[:-1],linmod(ybins[:-1],*popt),c='k',lw=.2)
    print(popt)

ax[2].set_ylim([-.1,.1])

In [ ]:
def temp(x, a, b, c):
    
    cold = np.maximum(0,a*(x-b))
    warm = -c*(x-273.15)

    return np.minimum(cold,warm)

popt_t, pcov = curve_fit(temp, 
                       xbins[:-1], popts, 
                       p0=[0.001, 220,.01])

print(popt_t)

fig,ax = plt.subplots(1,2)
ax[0].plot(xbins[:-1],popts)
ax[0].axvline(273.15,0,1,c='k',ls=':')
ax[0].axhline(0,0,1,c='k',ls=':')
ax[0].plot(xbins[:-1],temp(xbins[:-1],*popt_t))

In [ ]:
def func(data, a, b, c):
    U,T,S = data
    ampl = temp(T, a, b, c)
    return ampl*U**popt_ws[1]*S**popt_ws[2]

print(popt_t)
print(popt_ws[1:])

subl_pred = func((u10abs_valid,tas_valid,slope_valid),*popt_t)
#subl_pred = func((u10abs_valid,tas_valid,slope_valid),4e-4,224,1e-2)

In [ ]:
fig,ax = plt.subplots(1,4, figsize=(10,3),sharey=True)

h, xedges, yedges, im = ax[0].hist2d(
    u10abs_valid, subl_valid,
    bins=(300, 300),          # Resolution
    cmap='cmc.batlow_r',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6),
    vmin=None               # Auto-scaling
)

h, xedges, yedges, im = ax[1].hist2d(
    tas_valid, subl_valid,
    bins=(300, 300),          # Resolution
    cmap='cmc.batlow_r',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6),
    vmin=None               # Auto-scaling
)

h, xedges, yedges, im = ax[2].hist2d(
    slope_valid, subl_valid,
    bins=(300, 300),          # Resolution
    cmap='cmc.batlow_r',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6),
    vmin=None               # Auto-scaling
)

h, xedges, yedges, im = ax[3].hist2d(
    subl_pred, subl_valid,
    bins=(300, 300),          # Resolution
    cmap='cmc.batlow_r',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6),
    vmin=None               # Auto-scaling
)

ax[3].plot([-1,1],[-1,1],c='r')

ax[0].set_ylim(-.025,.06)
ax[3].set_xlim(-.025,.06)

In [ ]:
ds = xr.open_mfdataset('/home/erwin/data/dalum_racmo241/subltot_*.nc')
subl = -ds.subltot.isel(height=0).mean(dim='time')*12. # mm w.e. /y
ds.close()

ds = xr.open_dataset('../results/t1p1/main_output_ANT_grid.nc')
M = ds.Sublimation.sum(dim='month').isel(time=-1) * 1000 # Integrate to mm w.e./y
umask = ds.Hi.isel(time=-1).values>0
ux = ds.x.values
uy = ds.y.values
ds.close()

data_crs = ccrs.PlateCarree()
proj = ccrs.Stereographic(central_latitude=-90, central_longitude=0)

fig,ax = plt.subplots(1,3,figsize=(11,3))

h, xedges, yedges, im = ax[0].hist2d(
    subl_pred*1000., subl_valid*1000.,
    bins=(100, 100),
    cmap='afmhot_r',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6)
)
ax[0].plot([-600,600],[-600,600],c='r',lw=1)
ax[0].set_xlabel('F(Tas, U10m, Hs_slope) [mmwe/month]')
ax[0].set_ylabel('Subl RACMO [mmwe/month]')
ax[0].set_title('Sublimation')
plt.colorbar(im,ax=ax[0],label='Count')

bounds = [-500,-300,-150,-75,-25,-10,-3,0,3,10,25,75,150,300,500]

cmap = plt.get_cmap('RdBu_r')
norm = mpl.colors.BoundaryNorm(bounds,cmap.N)

ax1 = fig.add_subplot(1,3,2,projection=proj)
ax1.set_extent([-180,180,-90,-45], crs=data_crs)
im = ax1.pcolormesh(lon,lat,np.where(mask,subl,np.nan),cmap=cmap,norm=norm, transform=data_crs)

im = ax[2].pcolormesh(ux,uy,np.where(umask,M,np.nan),cmap=cmap,norm=norm)
cb = plt.colorbar(im,ax=ax[2],label='Sublimation [mm w.e. / yr]',extend='both')
ax[1].set_title('RACMO: 236.2 Gt/y')
Mtot = np.nansum(M)*16*16*1e-6
ax[2].set_title(f'ITM: {Mtot:.1f} Gt/y')

ax[0].set_xlim([-100,100])
ax[0].set_xlim([-100,100])

for Ax in ax:
    Ax.set_aspect(1)

for Ax in [ax1,ax[2]]:
    Ax.set_xlim([-3e6,3e6])
    Ax.set_ylim([-3e6,3e6])

for Ax in ax[1:]:
    Ax.set_xticks([])
    Ax.set_yticks([])

plt.tight_layout()
plt.savefig('../figures/draftplot_ITM_subl.png',dpi=600)